<a href="https://colab.research.google.com/github/gibsonx/jlpt_simulator/blob/dev/graphs/n3/outliner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
if 'google.colab' in str(get_ipython()):
    !git clone https://github.com/gibsonx/jlpt_simulator.git
    %cd jlpt_simulator
    !git checkout dev
    !apt-get install python3-dev graphviz libgraphviz-dev pkg-config
    !pip install -r requirements.txt
else:
  print('Not running on CoLab')

Not running on CoLab


In [2]:
import json
import logging
import random
import time
import pandas as pd
import yaml
import inspect
from tqdm import tqdm
import os
from libs.Logger import logger
from datetime import datetime
from docx import Document
from html4docx import HtmlToDocx
import uuid
from libs.CosmosMongoDB import CosmosMongoDB
from libs.LLMs import *
from IPython.display import display, Markdown, HTML
import datetime
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
from libs.Utils import render_to_html,collect_vocabulary,_load_vocab_and_resources
from graphs.common.Schema import Outline
from langchain_core.prompts import ChatPromptTemplate
from graphs.common.ExamGenerator import ExamGenerator
load_dotenv()

from graphs.common.TaskRunner import TaskRunner

# N1 Level Exam

In [3]:
# runner = TaskRunner(level="N1", exam_type="fast_exam")
# n1_outline, n1_exam_paper = runner.run()

## N1 Outline Preview

In [4]:
# display(Markdown(n1_outline.as_str))

## N1 HTML Result

In [5]:
# html_output = render_to_html(n1_exam_paper['sections'])
# display(HTML(html_output))
# timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
# filename = f"./output/jlpt_simulator/JLPT_{timestamp}.html"

# with open(filename, "w", encoding="utf-8") as file:
#     file.write(html_output)

# N2 Level Exam

In [6]:
runner = TaskRunner(level="N2", exam_type="full_exam")
n2_outline, n2_exam_paper = runner.run()

2025-11-09 22:05:05,809 - INFO - jlpt - Module 'graphs.n2.outliner' imported successfully.
2025-11-09 22:05:05,809 - INFO - Module 'graphs.n2.outliner' imported successfully.
2025-11-09 22:05:18,962 - INFO - HTTP Request: POST https://ai-rolandaws880125ai409947751408.openai.azure.com/openai/deployments/gpt-4.1/chat/completions?api-version=2025-01-01-preview "HTTP/1.1 200 OK"
2025-11-09 22:05:19,045 - INFO - jlpt - Outline of the exam:

# 日本語能力試験N2 模擬試験問題

## 第1部：語彙

### kanji_reading

問題1：ことばの読み方として最もよいものを、1・2・3・4から一つえらびなさい（名詞1問、80%難易度：非常に難しい）

- **ひっしゃ**

### write_kanji

問題2：このことばを漢字で書くとき、最もよいものを、1・2・3・4から一つえらびなさい（名詞1問、80%難易度：非常に難しい）

- **つうち**

### words_collocation

問題3：（　）に入れるのに最もよいものを、1・2・3・4から一つえらびなさい（動詞1問、80%難易度：非常に難しい）

- **ひっかかる**

### word_meaning

問題4：（　）に入れるのに最もよいものを、1・2・3・4から一つえらびなさい（動詞1問、80%難易度：非常に難しい）

- **ふるまう**

### synonym_substitution

問題5：意味が最も近いものを、1・2・3・4から一つえらびなさい（動詞1問、80%難易度：非常に難しい）

- **ぞうげん**

### word_usage

問題6：つぎのことばの使い方として最もよいものを、1・2・3・4から一つえらびなさい（動詞1問、80

## N2 Outline Preview

In [7]:
display(Markdown(n2_outline.as_str))

# 日本語能力試験N2 模擬試験問題

## 第1部：語彙

### kanji_reading

問題1：ことばの読み方として最もよいものを、1・2・3・4から一つえらびなさい（名詞1問、80%難易度：非常に難しい）

- **ひっしゃ**

### write_kanji

問題2：このことばを漢字で書くとき、最もよいものを、1・2・3・4から一つえらびなさい（名詞1問、80%難易度：非常に難しい）

- **つうち**

### words_collocation

問題3：（　）に入れるのに最もよいものを、1・2・3・4から一つえらびなさい（動詞1問、80%難易度：非常に難しい）

- **ひっかかる**

### word_meaning

問題4：（　）に入れるのに最もよいものを、1・2・3・4から一つえらびなさい（動詞1問、80%難易度：非常に難しい）

- **ふるまう**

### synonym_substitution

問題5：意味が最も近いものを、1・2・3・4から一つえらびなさい（動詞1問、80%難易度：非常に難しい）

- **ぞうげん**

### word_usage

問題6：つぎのことばの使い方として最もよいものを、1・2・3・4から一つえらびなさい（動詞1問、80%難易度：非常に難しい）

- **さまたげる**

## 第2部：文法

### sentence_grammar

問題1：つぎの文の（　　　）に入れるのに最もよいものを、１・２・３・４から一つえらびなさい（副詞1問、トピック：交通状況について話す）

- **交通状況について話す**どうやら（どうやら）

### sentence_sort

問題2：つぎの文の ★ に入る最もよいものを、1・2・3・4から一つえらびなさい（2問、トピック：健康診断や医者への訪問について話す、家事の分担について話す）

- **健康診断や医者への訪問について話す**～かねる（かねる）
- **家事の分担について話す**ばかりか（ばかりか）

### sentence_structure

問題3：つぎの文章を読んで、文章全体の内容を考えて、文中の 48 から 51 の中に入る最もよいものを、1・2・3・4から一つえらびなさい（1問、トピック：趣味について話す）

- **趣味について話す**～ふうに（ふうに）

## 第3部：読解

### short_passage_narrative_read

問題1-1：つぎの文章を読んで、質問に答えなさい。答えは、1・2・3・4から最もよいものを一つえらびなさい（1記事、トピック：引っ越しの準備について話す）

- **引っ越しの準備について話す**

### short_passage_mail_read

問題1-2：つぎの文章を読んで、質問に答えなさい。答えは、1・2・3・4から最もよいものを一つえらびなさい（1記事、トピック：領収書を求める）

- **領収書を求める**

### short_passage_notification_read

問題1-3：つぎの文章を読んで、質問に答えなさい。答えは、1・2・3・4から最もよいものを一つえらびなさい（1記事、トピック：週末の予定について話す）

- **週末の予定について話す**

### midsize_passage_read

問題2：つぎの(1)と(2)の文章を読んで、質問に答えなさい。答えは、1・2・3・4から最もよいものを一つえらびなさい（1記事、トピック：日本の祭りや文化イベントについて話す）

- **日本の祭りや文化イベントについて話す**

### comprehensive_reading

問題3：つぎの(1)と(2)の文章を読んで、質問に答えなさい。答えは、1・2・3・4から最もよいものを一つえらびなさい（1記事、トピック：技術について話す）

- **技術について話す**

### long_passage_read

問題4：つぎの文章を読んで、質問に答えなさい。答えは、1・2・3・4から最もよいものを一つえらびなさい（1記事、トピック：環境問題について話す）

- **環境問題について話す**

### info_retrieval

問題5：これを読んで、下の質問に答えなさい。答えは、1・2・3・4から最もよいものを一つえらびなさい（1記事、トピック：公共施設の利用方法について話す）

- **公共施設の利用方法について話す**

## 第4部：聴解

### topic_understanding

問題1：まず質問を聞いてください。それから話を聞いて、問題用紙の1から4の中から、最もよいものを一つえらんでください（1問、トピック：おすすめを尋ねる）

- **おすすめを尋ねる**

### keypoint_understanding

問題2：まず質問を聞いてください。そのあと、問題用紙を見てください。読む時間があります。それから話を聞いて、問題用紙の1から4の中から、最もよいものを一つえらんでください（1問、トピック：購入したい商品の説明）

- **購入したい商品の説明**

### summary_understanding

問題3：問題用紙に何もいんさつされていません。この問題は、ぜんたいとしてどんな内容かを聞く問題です。話の前に質問はありません。まず話を聞いてください。それから、質問と選択肢を聞いて、1から4の中から、最もよいものを一つえらんでください（1問、トピック：家族について話す）

- **家族について話す**

### immediate_ack

問題4：問題用紙に何もいんさつしていません。まず文を聞いてください。それから、その返事を聞いて、1から3の中から、最もよいものを一つえらんでください（1問、トピック：レストランで食べ物を注文する）

- **レストランで食べ物を注文する**

### comprehensive_expression_listen_answer

問題5-1：長めの話を聞きます。この問題には練習はありません。問題用紙にメモをとってもかまいません（1問、トピック：仕事のプロジェクトについて話す）

- **仕事のプロジェクトについて話す**

### comprehensive_expression_show_answer

問題5-2：長めの話を聞きます。この問題には練習はありません。問題用紙にメモをとってもかまいません（1問、トピック：旅行の計画について話す）

- **旅行の計画について話す**

## N2 Exam Result

In [8]:
html_output = render_to_html(n2_exam_paper['sections'])
display(HTML(html_output))

開館日,開館時間,休館日
月～土,9:00～19:00,日曜・祝日・毎月最終木曜
区分,利用時間,利用料
午前,9:00～12:00,"3,000円"
午後,13:00～17:00,"4,500円"
夜間,18:00～21:00,"5,000円"


In [9]:
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"./output/JLPT_N2_{timestamp}.html"

with open(filename, "w", encoding="utf-8") as file:
    file.write(html_output)

# N3 Level Exam

In [10]:
runner = TaskRunner(level="N3", exam_type="fast_exam")
n3_outline, n3_exam_paper = runner.run()

## N3 Outline Preview

In [11]:
display(Markdown(n3_outline.as_str))

## N3 Exam Result

In [12]:
html_output = render_to_html(n3_exam_paper['sections'])
display(HTML(html_output))

In [13]:
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"./output/JLPT_N3_{timestamp}.html"

with open(filename, "w", encoding="utf-8") as file:
    file.write(html_output)

# N4 Level Exam

In [14]:
# runner = TaskRunner(level="N4", exam_type="fast_exam")
# n4_outline, n4_exam_paper = runner.run()

## N4 Outline Preview

In [15]:
# display(Markdown(n4_outline.as_str))

## N4 Exam Result

In [16]:
# html_output = render_to_html(n4_exam_paper['sections'])
# display(HTML(html_output))

# N5 Level Exam

In [17]:
# runner = TaskRunner(level="N5", exam_type="fast_exam")
# n5_outline, n5_exam_paper = runner.run()

## N5 Outline Preview

In [18]:
# display(Markdown(n5_outline.as_str))

## N5  Exam Result

In [19]:
# html_output = render_to_html(n5_exam_paper['sections'])
# display(HTML(html_output))